# TPS · Módulo 1 — Contextualización y verificación del preprocesamiento
**PSIB 2026Q2 · Grupo 12** — Lautersztein, Reggio, Ballester
*Caracterización de la carga cognitiva mediante análisis espectral del EEG durante aritmética mental*

Este notebook cubre las etapas 1 a 3 del pipeline del anteproyecto:

1. **Contextualización**: qué hay en la base (sujetos, canales, fs, duración, amplitudes).
2. **Verificación del preprocesamiento**: con la PSD de Welch comprobamos si los filtros que declaran los autores (pasa-altos 0,5 Hz, pasa-bajos 45 Hz, notch 50 Hz) realmente están aplicados, y con eso decidimos si hace falta filtrar de nuevo.
3. **Segmentación temporal**: 60 s por condición, cortados en épocas de 4 s (las *realizaciones* del proceso), con un chequeo exploratorio de estacionariedad.

La salida (`modulo1_epocas.npz`) es la entrada del Módulo 2 (análisis de ritmos).

## 0. Instalación y descarga
Base: *EEG During Mental Arithmetic Tasks* (PhysioNet, `eegmat` v1.0.0). Son 36 sujetos × 2 archivos EDF:
`SubjectNN_1.edf` = **reposo** (ojos cerrados) y `SubjectNN_2.edf` = **cálculo mental** (restas seriadas).
`subject-info.csv` tiene la calidad de la cuenta de cada sujeto (1 = buen desempeño → grupo G, 0 = bajo → grupo B).

In [ ]:
!pip -q install mne
# Descarga toda la base en la carpeta eegmat/ (-N: no vuelve a bajar lo que ya está)
!wget -q -r -N -c -np -nH --cut-dirs=3 -P eegmat https://physionet.org/files/eegmat/1.0.0/

import numpy as np, pandas as pd, matplotlib.pyplot as plt, mne
from scipy import signal
mne.set_log_level('ERROR')

## 1. Contextualización
### 1.1 Sujetos y grupos de desempeño

In [ ]:
info = pd.read_csv('eegmat/subject-info.csv')
col_calidad = [c for c in info.columns if 'quality' in c.lower()][0]   # columna "Count quality"
info['grupo'] = np.where(info[col_calidad] == 1, 'G', 'B')            # G = buen desempeño, B = bajo
print(info.grupo.value_counts().to_dict())
info.head()

### 1.2 Carga de los registros
Nos quedamos con los **19 canales del sistema 10-20** (descartamos la derivación A2-A1 y el ECG que también vienen en el EDF).
MNE devuelve la señal en voltios: la pasamos a **µV**, que es la escala habitual del EEG.

In [ ]:
CANALES = ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'Fz', 'C3', 'C4', 'Cz',
           'P3', 'P4', 'Pz', 'O1', 'O2', 'T3', 'T4', 'T5', 'T6']
COND = {1: 'reposo', 2: 'tarea'}

def cargar(sujeto, cond):
    """Lee un EDF y devuelve (datos [canales x muestras] en µV, fs) con los canales en el orden de CANALES."""
    raw = mne.io.read_raw_edf(f'eegmat/{sujeto}_{cond}.edf', preload=True)
    nombres = [c.replace('EEG', '').strip() for c in raw.ch_names]     # 'EEG Fp1' -> 'Fp1'
    idx = [nombres.index(c) for c in CANALES]
    return raw.get_data()[idx] * 1e6, raw.info['sfreq']

eeg = {(s, c): cargar(s, c) for s in info.Subject for c in COND}       # diccionario {(sujeto, condición): (x, fs)}

### 1.3 Montaje, frecuencia de muestreo, duración y amplitud
La tabla resume, por condición, la fs, la duración y dos medidas de amplitud: el **RMS** (nivel típico de la señal) y el **pico a pico máximo** entre canales (valores muy grandes sugieren artefactos residuales).

In [ ]:
# Posición de los 19 electrodos en el sistema 10-20
info_mne = mne.create_info(CANALES, 500, 'eeg').set_montage('standard_1020')
mne.viz.plot_sensors(info_mne, show_names=True)

# Filtros declarados en el encabezado del EDF (el README dice "pasa-altos 30 Hz", que no coincide)
hdr = mne.io.read_raw_edf(f'eegmat/{info.Subject[0]}_1.edf').info
print(f"Encabezado EDF -> pasa-altos: {hdr['highpass']} Hz · pasa-bajos: {hdr['lowpass']} Hz")

ctx = pd.DataFrame([dict(sujeto=s, condicion=COND[c], fs=fs, duracion_s=x.shape[1] / fs,
                         rms_uV=x.std(axis=1).mean(), ptp_max_uV=np.ptp(x, axis=1).max())
                    for (s, c), (x, fs) in eeg.items()])
ctx.groupby('condicion')[['fs', 'duracion_s', 'rms_uV', 'ptp_max_uV']].agg(['min', 'median', 'max']).round(1)

### 1.4 Inspección visual de la señal
10 s de reposo y de tarea del mismo sujeto, con los canales desplazados verticalmente. Lo que buscamos: amplitudes razonables (decenas de µV), ausencia de saltos o canales planos, y a simple vista el ritmo alfa occipital (O1/O2) en reposo con ojos cerrados.

In [ ]:
SUJ, T = info.Subject[0], 10                      # sujeto de ejemplo y segundos a graficar
fig, ax = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
for a, c in zip(ax, COND):
    x, fs = eeg[SUJ, c]
    t = np.arange(int(T * fs)) / fs
    a.plot(t, x[:, :len(t)].T - 100 * np.arange(len(CANALES)), lw=0.5, color='k')   # 100 µV entre canales
    a.set(title=f'{SUJ} · {COND[c]}', xlabel='Tiempo [s]')
ax[0].set_yticks(-100 * np.arange(len(CANALES)), CANALES)
plt.tight_layout()

Pico a pico por canal (todos los sujetos). Los canales frontales (Fp1, Fp2, F7, F8) son los más expuestos a artefactos oculares y los temporales (T3–T6) a los musculares: si alguno se despega del resto es una pista de que el ICA de los autores no los eliminó por completo.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5), sharey=True)
for a, c in zip(ax, COND):
    a.boxplot(np.array([np.ptp(x, axis=1) for (s, cc), (x, fs) in eeg.items() if cc == c]))   # [sujetos x canales]
    a.set_xticks(range(1, len(CANALES) + 1), CANALES)
    a.set(title=f'Pico a pico por canal · {COND[c]}', ylabel='µV')
plt.tight_layout()

## 2. Verificación del preprocesamiento
Estimamos la PSD con **Welch**: ventanas de Hamming con 50 % de solapamiento. Promediar los periodogramas de varias ventanas reduce la varianza del estimador, que es lo que necesitamos para ver la forma del espectro.

Para esta verificación usamos **ventanas de 8 s** (resolución Δf = 1/8 = 0,125 Hz). Con las ventanas de 2 s del análisis principal (Δf = 0,5 Hz) no hay ningún punto del espectro entre 0 y 0,5 Hz, y no se podría ver el pasa-altos. El costo es promediar menos ventanas (más varianza), pero se compensa porque además promediamos entre canales y sujetos.

Si los filtros declarados están aplicados deberíamos ver:
- caída de potencia **por debajo de 0,5 Hz** (pasa-altos),
- caída fuerte **por encima de 45 Hz** (pasa-bajos),
- **ningún pico en 50 Hz** (notch / interferencia de línea).

In [ ]:
def psd_welch(x, fs, ventana_s=2):
    """PSD por Welch (Hamming, 50 % de solapamiento) sobre el último eje. Devuelve (f [Hz], PSD [µV²/Hz])."""
    n = int(ventana_s * fs)
    return signal.welch(x, fs, window='hamming', nperseg=n, noverlap=n // 2, axis=-1)

f = psd_welch(*eeg[SUJ, 1], ventana_s=8)[0]
psd = {c: np.array([psd_welch(*eeg[s, c], ventana_s=8)[1] for s in info.Subject]) for c in COND}   # [sujetos x canales x f]

fig, ax = plt.subplots(figsize=(10, 4))
for c in COND:
    ax.semilogy(f, psd[c].mean(axis=(0, 1)), label=COND[c])            # promedio entre sujetos y canales
for fc in (0.5, 45, 50):
    ax.axvline(fc, ls='--', c='gray', lw=0.8)
ax.set(xlabel='Frecuencia [Hz]', ylabel='PSD [µV²/Hz]', title='PSD promedio (Welch)', xlim=(0, 100))
ax.legend();

Para no depender sólo de la vista, cuantificamos tres indicadores por registro (en dB):
- **baja_dB**: potencia en 0,1–0,5 Hz respecto de 0,5–4 Hz → negativo si el pasa-altos actúa. Se excluye 0 Hz porque Welch le resta la media a cada ventana.
- **alta_dB**: potencia en 45–100 Hz respecto de 0,5–45 Hz → muy negativo si el pasa-bajos actúa.
- **pico50_dB**: PSD en 50 Hz respecto de la banda 40–45 Hz, justo antes de los filtros → muy negativo si el notch eliminó la interferencia de red.

In [ ]:
banda = lambda P, f1, f2: P[..., (f >= f1) & (f < f2)].mean(axis=-1)    # potencia media en [f1, f2)
dB = lambda r: 10 * np.log10(r)

filas = []
for c in COND:
    P = psd[c].mean(axis=1)                                             # promedio entre canales -> [sujetos x f]
    filas.append(pd.DataFrame(dict(sujeto=info.Subject, condicion=COND[c],
        baja_dB=dB(banda(P, 0.1, 0.5) / banda(P, 0.5, 4)),
        alta_dB=dB(banda(P, 45, 100) / banda(P, 0.5, 45)),
        pico50_dB=dB(P[:, f == 50][:, 0] / banda(P, 40, 45)))))
ver = pd.concat(filas)
ver.groupby('condicion')[['baja_dB', 'alta_dB', 'pico50_dB']].agg(['mean', 'min', 'max']).round(1)

PSD en dB por canal (promedio entre sujetos, reposo): sirve para detectar canales con exceso de baja frecuencia (oculares, frontales) o de alta frecuencia (musculares, temporales).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(10 * np.log10(psd[1].mean(axis=0)), aspect='auto', origin='lower',
               extent=[f[0], f[-1], -0.5, len(CANALES) - 0.5], cmap='viridis')
ax.set(yticks=range(len(CANALES)), yticklabels=CANALES, xlim=(0, 60), xlabel='Frecuencia [Hz]', title='PSD por canal · reposo [dB]')
fig.colorbar(im, label='dB');

### Resultado de la verificación y decisión sobre el filtrado
*(Corrida sobre los 36 sujetos.)*

- **Pasa-altos 0,5 Hz: confirmado.** La potencia en 0,1–0,5 Hz está en promedio 5–6 dB por debajo de la de 0,5–4 Hz. La caída es suave, lo que indica un filtro de orden bajo.
- **Pasa-bajos 45 Hz: confirmado.** Por encima de 45 Hz la potencia cae ~31 dB respecto de la banda útil. La caída más profunda (≈ 47–53 Hz) es la del notch. Entre 55 y 70 Hz queda algo de potencia (pasa-bajos de orden bajo), pero está fuera de las bandas que se analizan (hasta 30 Hz).
- **Notch 50 Hz: funciona en casi todos los registros.** En promedio, 50 Hz queda 17–24 dB por debajo de la banda 40–45 Hz. En el reposo de 2 sujetos (Subject12 y Subject22) queda un residuo de red del mismo nivel que esa banda. Como 50 Hz está fuera de las bandas de interés, no afecta las potencias por banda.
- **Decisión: no se vuelve a filtrar.** Filtrar dos veces sólo agregaría distorsión de fase y de amplitud sin beneficio. La comparación de filtros FIR vs. IIR del anteproyecto queda como análisis de sensibilidad opcional.
- Ya en esta etapa se ve el efecto esperado: el **pico alfa (~10 Hz) es más alto en reposo que durante la tarea.**
- Como el ICA no puede replicarse en el TP, en el informe se discute qué se habría visto sin él: artefactos oculares inflando delta y theta frontales, y musculares inflando beta temporal.

## 3. Segmentación temporal
- El reposo dura más que la tarea: para que ambas condiciones tengan **igual duración** tomamos **60 s por condición** (en el reposo, los últimos 60 s, los más cercanos en el tiempo a la tarea).
- Esos 60 s se cortan en **épocas de 4 s sin solapamiento** → 15 realizaciones por canal, cada una lo bastante corta para considerarla aproximadamente estacionaria.
- **Estacionariedad (exploratorio)**: si el proceso es estacionario, sus estadísticos no deberían cambiar entre épocas. La media no aporta información, porque los filtros ya la llevan a ≈ 0, así que miramos la **varianza**. Medimos cuánto varía entre épocas (`cv_varianza`) y marcamos como **atípica** una época cuya varianza supera 3 veces la mediana del canal (cambio abrupto, posible artefacto residual).

In [ ]:
DUR_SEG, DUR_EPOCA = 60, 4     # segundos

def segmentar(x, fs):
    """Últimos DUR_SEG s del registro cortados en épocas de DUR_EPOCA s -> [épocas x canales x muestras]."""
    x, m = x[:, -int(DUR_SEG * fs):], int(DUR_EPOCA * fs)
    k = x.shape[1] // m                                   # cantidad de épocas completas
    return x[:, :k * m].reshape(len(x), k, m).transpose(1, 0, 2)

epocas = {(s, c): segmentar(*eeg[s, c]) for s, c in eeg}

var ={k: ep.var(axis=-1) for k, ep in epocas.items()}                    # varianza por época y canal -> [épocas x canales]
atipica = {k: (v > 3 * np.median(v, axis=0)).any(axis=1) for k, v in var.items()}   # época atípica si lo es en algún canal
est = pd.DataFrame([dict(sujeto=s, condicion=COND[c], n_epocas=len(v),
                         cv_varianza=(v.std(0) / v.mean(0)).mean(),         # cuánto cambia la varianza entre épocas
                         epocas_atipicas=atipica[s, c].sum())
                    for (s, c), v in var.items()])
print(f"Épocas atípicas: {np.mean(np.concatenate(list(atipica.values()))):.1%} del total")
est.groupby('condicion')[['n_epocas', 'cv_varianza', 'epocas_atipicas']].agg(['mean', 'min', 'max']).round(2)

In [ ]:
# Varianza por época y canal del sujeto de ejemplo: una época que "se enciende" en muchos canales es un cambio abrupto
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for a, c in zip(ax, COND):
    im = a.imshow(var[SUJ, c].T, aspect='auto', cmap='magma')                # [canales x épocas]
    a.set(title=f'{SUJ} · {COND[c]} · varianza por época [µV²]', xlabel='Época (4 s)', yticks=range(len(CANALES)), yticklabels=CANALES)
    fig.colorbar(im, ax=a)
plt.tight_layout()

## 4. Guardado para el Módulo 2
Se guardan las épocas de cada sujeto y condición, la marca de épocas atípicas (para excluirlas en el Módulo 2) y la tabla de sujetos. En Colab los archivos se borran al cerrar la sesión: para conservarlos, montar Drive (`from google.colab import drive; drive.mount('/content/drive')`) y cambiar la ruta.

In [ ]:
np.savez_compressed('modulo1_epocas.npz', canales=CANALES, fs=eeg[SUJ, 1][1],
                    **{f'{s}_{COND[c]}': ep for (s, c), ep in epocas.items()},
                    **{f'{s}_{COND[c]}_atipicas': a for (s, c), a in atipica.items()})
info.to_csv('modulo1_sujetos.csv', index=False)

## 5. Conclusiones del Módulo 1
*(Corrida sobre los 36 sujetos.)*

**Base de datos**
- 36 sujetos, 19 canales del 10-20, fs = 500 Hz en todos los registros. El encabezado EDF confirma el filtrado 0,5–45 Hz. El README dice "pasa-altos de 30 Hz", que es un error.
- **Grupos de desempeño:** según `subject-info.csv` hay **26 sujetos G y 10 B**, no 24 y 12 como dicen el README y el anteproyecto. Se usa lo que figura en el archivo, y hay que corregir el número en el informe.
- **Duraciones:** reposo ≈ 182 s y tarea 62 s. Un sujeto (Subject31) tiene sólo 80 s de reposo. Todos tienen al menos 60 s, así que la segmentación es pareja para todos.
- **Amplitud:** RMS típico ~10 µV. En 6 sujetos hay canales con picos > 300 µV (máximo ~790 µV), que son artefactos residuales puntuales.

**Preprocesamiento:** los filtros declarados están aplicados y no se re-filtra (ver sección 2).

**Segmentación y estacionariedad**
- 60 s por condición → 15 épocas de 4 s por registro.
- La varianza cambia en promedio ~37 % entre épocas, así que el EEG es sólo **aproximadamente estacionario**. Eso justifica estimar la PSD promediando sobre épocas (Welch) en vez de sobre el registro entero.
- El **10 %** de las épocas resulta atípico: en promedio 1,5 de 15 por registro. Se guardan marcadas para **excluirlas** en el Módulo 2 y comparar el resultado con y sin ellas.